# Model Configuration Comparison Report

**Purpose**: Compare performance across different training configurations

**Configs**: Baseline, Debug, Small, Large

In [ ]:
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

config_dir = Path('configs/experiments')
configs = {}

for config_file in ['baseline.yaml', 'debug.yaml', 'cnn_small.yaml', 'cnn_large.yaml']:
    with open(config_dir / config_file, 'r') as f:
        configs[config_file.replace('.yaml', '')] = yaml.safe_load(f)

print("✓ Configurations loaded:")
for name in configs.keys():
    print(f"  • {name}")

In [ ]:
# Extract key parameters
comparison_data = []

for config_name, config in configs.items():
    comparison_data.append({
        'Configuration': config_name.replace('_', ' ').title(),
        'Epochs': config['training'].get('epochs', 'N/A'),
        'Batch Size': config['training'].get('batch_size', 'N/A'),
        'Learning Rate': config['optimizer'].get('lr', 'N/A'),
        'Hidden Features': config['model'].get('hidden_features', 'N/A'),
        'Weight Decay': config['optimizer'].get('weight_decay', 0.0),
        'Optimizer': config['optimizer'].get('name', 'N/A'),
        'Early Stopping': config['training'].get('early_stopping_patience', 'N/A')
    })

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*120)
print("📋 Configuration Comparison")
print("="*120)
print(comparison_df.to_string(index=False))
print("="*120)

In [ ]:
# Simulated expected performance
expected_performance = pd.DataFrame({
    'Configuration': ['Baseline', 'Debug', 'Small', 'Large'],
    'Expected Val Acc': [0.9875, 0.9750, 0.9810, 0.9920],
    'Est. Training Time (min)': [15, 2, 8, 28],
    'Model Size (MB)': [0.84, 0.68, 0.68, 1.20],
    'Use Case': [
        'Production',
        'Testing/Development',
        'Fast Training',
        'Max Accuracy'
    ]
})

print("\n" + "="*100)
print("📊 Expected Performance")
print("="*100)
print(expected_performance.to_string(index=False))
print("="*100)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

configs_list = expected_performance['Configuration']
x = np.arange(len(configs_list))

# Accuracy
axes[0, 0].bar(x, expected_performance['Expected Val Acc'], 
               color=['steelblue', 'orange', 'green', 'red'], alpha=0.7, edgecolor='black')
axes[0, 0].set_ylabel('Validation Accuracy', fontsize=11)
axes[0, 0].set_title('Expected Accuracy Comparison', fontsize=12, fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(configs_list)
axes[0, 0].set_ylim([0.96, 1.0])
axes[0, 0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(expected_performance['Expected Val Acc']):
    axes[0, 0].text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')

# Training time
axes[0, 1].barh(configs_list, expected_performance['Est. Training Time (min)'],
                color=['steelblue', 'orange', 'green', 'red'], alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Training Time (minutes)', fontsize=11)
axes[0, 1].set_title('Estimated Training Time', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='x')
for i, v in enumerate(expected_performance['Est. Training Time (min)']):
    axes[0, 1].text(v + 0.5, i, f'{v}m', fontsize=9, fontweight='bold')

# Hidden features (from configs)
hidden_features = [config['model']['hidden_features'] for config in configs.values()]
axes[1, 0].bar(x, hidden_features,
              color=['steelblue', 'orange', 'green', 'red'], alpha=0.7, edgecolor='black')
axes[1, 0].set_ylabel('Hidden Features', fontsize=11)
axes[1, 0].set_title('Model Capacity (Hidden Features)', fontsize=12, fontweight='bold')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(configs_list)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Trade-off plot
colors_map = {'Baseline': 'steelblue', 'Debug': 'orange', 'Small': 'green', 'Large': 'red'}
for idx, row in expected_performance.iterrows():
    axes[1, 1].scatter(row['Est. Training Time (min)'], row['Expected Val Acc'],
                      s=500, alpha=0.7, color=colors_map[row['Configuration']], edgecolor='black', linewidth=2)
    axes[1, 1].annotate(row['Configuration'], 
                       (row['Est. Training Time (min)'], row['Expected Val Acc']),
                       fontsize=10, ha='center', fontweight='bold')

axes[1, 1].set_xlabel('Training Time (minutes)', fontsize=11)
axes[1, 1].set_ylabel('Accuracy', fontsize=11)
axes[1, 1].set_title('Trade-off: Speed vs Accuracy', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim([0.96, 1.0])

plt.tight_layout()
plt.savefig('notebooks/assets/figures/report_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Comparison visualization saved")

In [ ]:
recommendations = pd.DataFrame({
    'Configuration': ['Baseline', 'Debug', 'Small', 'Large'],
    'Best For': [
        'Production - Balanced accuracy & speed',
        'Development - Quick testing & debugging',
        'Fast Training - Resource-constrained',
        'Maximum Accuracy - Offline/Batch processing'
    ],
    'When to Use': [
        'Default choice for deployment',
        'Rapid prototyping & testing',
        'Limited GPU/Memory or quick iterations',
        'When highest accuracy is critical'
    ]
})

print("\n" + "="*120)
print("🎯 Configuration Recommendations")
print("="*120)
for idx, row in recommendations.iterrows():
    print(f"\n{row['Configuration'].upper()}:")
    print(f"  📌 Best For: {row['Best For']}")
    print(f"  ✅ When to Use: {row['When to Use']}")
print("\n" + "="*120)